# 07 — Per-match per-player FIFA stats

Pulls from `fdh-api.fifa.com` (FIFA's internal data hub — no auth, just a `Referer: fifa.com` header):

- `/v1/stats/match/{IdIFES}/players.json` → ~100 raw stat fields per player per match (Goals, Assists, AttemptAtGoal*, BallProgressions, DefensivePressuresApplied, Distance*, LinebreaksAttempted, etc.)
- `/v1/powerranking/match/{IdIFES}.json` → per-player attacking/defensive/creativity rank + score, within-team rank.

Outputs:
- `wc26_player_match_stats` (long format: one row per `(fifa_match_id, fifa_player_id, stat_name)` × `value`)
- `wc26_player_match_powerrank` (wide: one row per `(fifa_match_id, fifa_player_id)`)

Matches whose stats haven't been published yet (404 from fdh) are skipped. Re-run with `force_refresh=True` to pull live updates.


In [1]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if (ROOT / "lib").is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "lib").is_dir():
    sys.path.insert(0, str(ROOT.parent))

from lib import io
from lib.http import polite_get

matches = io.load_table("wc26_matches")
candidates = matches.dropna(subset=["fifa_id_ifes"]).copy()
candidates["fifa_id_ifes"] = candidates["fifa_id_ifes"].astype("Int64").astype(str)
print(f"matches with fifa_id_ifes: {len(candidates)}/{len(matches)}")
print(f"  finished: {(candidates['status'] == 'finished').sum()}")


matches with fifa_id_ifes: 72/104
  finished: 43


## 1. Pull `players.json` per match (raw stats)


In [2]:
STATS_URL = "https://fdh-api.fifa.com/v1/stats/match/{ifes}/players.json"
POWER_URL = "https://fdh-api.fifa.com/v1/powerranking/match/{ifes}.json"

stats_rows = []
power_rows = []
skipped_404 = 0

for r in candidates.itertuples():
    ifes = r.fifa_id_ifes
    fifa_match_id = r.fifa_match_id

    # Player stats
    try:
        s = io.cache_raw(STATS_URL.format(ifes=ifes), source="fdh",
                         name=f"stats_match_{ifes}_players", sleep=0.15)
        for pid, stats in s.items():
            for entry in stats:
                if not isinstance(entry, list) or len(entry) < 2:
                    continue
                name, value = entry[0], entry[1]
                stats_rows.append({
                    "fifa_match_id": fifa_match_id,
                    "fifa_id_ifes": ifes,
                    "fifa_player_id": int(pid),
                    "stat_name": name,
                    "value": value,
                })
    except Exception as e:
        if "404" in str(e):
            skipped_404 += 1
        else:
            print(f"  stats err for {ifes}: {type(e).__name__}: {e}")

    # Power ranking
    try:
        p = io.cache_raw(POWER_URL.format(ifes=ifes), source="fdh",
                         name=f"powerranking_match_{ifes}", sleep=0.15)
        for grp in ("outfieldPlayers", "goalkeepers"):
            for pl in p.get(grp, []):
                power_rows.append({
                    "fifa_match_id": fifa_match_id,
                    "fifa_id_ifes": ifes,
                    "fifa_player_id": pl.get("playerId"),
                    "fifa_team_id": pl.get("teamId"),
                    "player_kind": grp[:-1],  # "outfieldPlayer" / "goalkeeper"
                    "attacking_rank": pl.get("attackingRank"),
                    "defensive_rank": pl.get("defensiveRank"),
                    "creativity_rank": pl.get("creativityRank"),
                    "attacking_score": pl.get("attackingScore"),
                    "defensive_score": pl.get("defensiveScore"),
                    "creativity_score": pl.get("creativityScore"),
                    "attacking_rank_within_team": pl.get("attackingRankWithinTeam"),
                    "defensive_rank_within_team": pl.get("defensiveRankWithinTeam"),
                    "creativity_rank_within_team": pl.get("creativityRankWithinTeam"),
                    "defending_the_goal_rank": pl.get("defendingTheGoalRank"),
                    "defending_the_goal_score": pl.get("defendingTheGoalScore"),
                    "in_possession_rank": pl.get("inPossessionRank"),
                    "in_possession_score": pl.get("inPossessionScore"),
                })
    except Exception as e:
        if "404" not in str(e):
            print(f"  power err for {ifes}: {type(e).__name__}: {e}")

print(f"\nstats rows: {len(stats_rows):,}")
print(f"power rows: {len(power_rows)}")
print(f"skipped 404 (match not yet played): {skipped_404}")



stats rows: 253,990
power rows: 1182
skipped 404 (match not yet played): 28


## 2. Build dataframes + filter to the curated stat allowlist

FDH publishes ~116 stat keys per match. We retain a curated 55 covering
shooting / passing / movement / defence / discipline / GK / set-pieces — enough
for the scoring engine without bloating the wide pivot.


In [3]:
# Curated 55 stat keys (shooting, ball progression, switches, distance/speed,
# pressures, distributions, set-pieces, GK, fouls, linebreaks, possession-sequence
# counts, offers/receptions, substitutions, discipline, basic outcomes).
STAT_ALLOWLIST = {
    "Assists", "AttemptAtGoal", "AttemptAtGoalOnTarget",
    "AttemptedBallProgressions", "AttemptedSwitchesOfPlay",
    "AvgSpeed", "CleanSheets",
    "CompletedBallProgressions", "CompletedSwitchesOfPlay",
    "Corners", "Crosses", "CrossesCompleted",
    "DefensivePressuresApplied",
    "DistanceHighSpeedSprinting", "DistanceWalking",
    "DistributionsCompletedUnderPressure", "DistributionsUnderPressure",
    "ForcedTurnovers", "FoulsAgainst", "FoulsFor", "FreeKicks",
    "GoalkeeperSaves", "Goals", "GoalsConceded", "GoalsOutsideThePenaltyArea",
    "LinebreaksAttempted", "LinebreaksAttemptedCompleted",
    "LinebreaksCompletedUnderPressure",
    "NumberOfInvolvements", "NumberOfPossessionSequences",
    "NumberOfShotEndingSequences",
    "OffersToReceiveTotal", "Offsides", "OwnGoals",
    "Passes", "PassesCompleted",
    "Penalties", "PenaltiesScored",
    "ReceivedOffersToReceive",
    "ReceptionsBetweenMidfieldAndDefensiveLine", "ReceptionsInBehind",
    "ReceptionsUnderNoPressure", "ReceptionsUnderPressure",
    "RedCards", "SpeedRuns", "Sprints",
    "SubstitutionsIn", "SubstitutionsOut",
    "TakeOnsCompleted", "TimePlayed", "TopSpeed", "TotalDistance",
    "XG", "YellowCards",
}

stats_df = pd.DataFrame(stats_rows)
power_df = pd.DataFrame(power_rows)

if len(stats_df):
    raw_keys = stats_df["stat_name"].nunique()
    stats_df = stats_df[stats_df["stat_name"].isin(STAT_ALLOWLIST)].reset_index(drop=True)
    stats_df["fifa_match_id"] = stats_df["fifa_match_id"].astype(str)
    n_players = stats_df.groupby("fifa_match_id")["fifa_player_id"].nunique().mean()
    n_stat_keys = stats_df["stat_name"].nunique()
    print(f"raw stat keys: {raw_keys}  →  kept after allowlist: {n_stat_keys}")
    print(f"distinct matches: {stats_df['fifa_match_id'].nunique()}")
    print(f"avg players per match: {n_players:.1f}")
    missing_from_data = STAT_ALLOWLIST - set(stats_df['stat_name'].unique())
    if missing_from_data:
        print(f"  (allowlist keys not present in this run's data: {sorted(missing_from_data)})")


raw stat keys: 116  →  kept after allowlist: 54
distinct matches: 44
avg players per match: 51.0


## 3. Sanity + save


In [4]:
# Quick join sanity: every fifa_player_id should exist in wc26_players.
players_master = io.load_table("wc26_player_enrichment")
known_ids = set(players_master["fifa_player_id"].dropna().astype(int))
if len(stats_df):
    unknown = stats_df.loc[~stats_df["fifa_player_id"].isin(known_ids), "fifa_player_id"].unique()
    print(f"unknown player_ids in stats (not in wc26_players): {len(unknown)}")
if len(power_df):
    unknown_p = power_df.loc[~power_df["fifa_player_id"].isin(known_ids), "fifa_player_id"].unique()
    print(f"unknown player_ids in power: {len(unknown_p)}")

if len(stats_df):
    io.save_table(stats_df, "wc26_player_match_stats")
if len(power_df):
    io.save_table(power_df, "wc26_player_match_powerrank")


unknown player_ids in stats (not in wc26_players): 2
unknown player_ids in power: 0


wrote data\processed\wc26_player_match_stats.parquet (118431 rows)
wrote data\processed\wc26_player_match_stats.csv
wrote data\processed\wc26_player_match_powerrank.parquet (1182 rows)
wrote data\processed\wc26_player_match_powerrank.csv
